# Data Preparation
This section describes how to organize raw CIFTI, GIFTI, and NIFTI data into the prepared NumPy arrays used by fMRI-HA. The pipeline validates the input files, records the validation results, writes `time x vertices` or `time x voxels` matrices, and optionally concatenates prepared files across runs or modules.

```{image} pic/data_prep/overall.png
:width: 800px
```

Most computational routines in fMRI-HA operate on data stored in NumPy (`.npy`) format. Prepared matrices are saved as `time points x vertices/voxels`, with separate folders for cortical hemispheres, CIFTI-derived subcortical regions, and NIFTI volume ROIs.

fMRI-HA follows a BIDS-inspired naming and directory convention. File names are constructed from key-value entities such as `sub`, `ses`, `run`, `space`, `hemi`, `roi`, `res`, `desc`, and `set`, which allows later stages to select files through simple filters.

Typical output hierarchy:

```text
work_dir/
  sub-<subject>/
    resample/
      response/
        hemi-L/
          sub-<subject>_ses-01_run-01_space-surf_hemi-L_roi-cortical_res-onavg32_desc-bold-demean_set-demo.npy
        hemi-R/
        volume-subcortical-L/
        volume-subcortical-R/
        volume-subcortical-BRAIN_STEM/
        volume-<roi_name>/
  logs/
    prep_log.json
    progress_<log_num>.log
  masks/
```

## Scripts Pipeline

Begin by importing the necessary packages.

In [ ]:
import joblib
import numpy as np
import neuroboros as nb
from pathlib import Path
from fmriha.preprocessing import prep

### (1) Input Validation

CIFTI and NIFTI files are validated with `prep.input_validation_nii`. GIFTI files are validated with `prep.input_validation_gii`. Both functions return a validation dictionary containing the subject list, session label, role label, input file metadata, grouped dimension information, missing files, and loading failures. This dictionary is the `input_info` argument used by the preprocessing functions below.

Parameters of `prep.input_validation_nii`:

| Parameter | Default | Meaning |
|---|---|---|
| `file_list` | Required | List of full CIFTI or NIFTI file paths. |
| `ses` | `"data"` | Session label written into output filenames as `ses-<label>`. |
| `bids_parse` | `True` | If `True`, parses BIDS-like filename entities such as `sub`, `run`, `task`, and `ses`. |
| `subject_key` | `"sub"` | Prefix used to locate subject IDs when `bids_parse=False`. |
| `check_dim` | `True` | If `True`, checks input image dimensions for consistency. |
| `group_by` | `"run"` | Filename entity used to group files before dimension checking. Use `None` to check all files together. |
| `role` | `"train"` | Dataset role written into output filenames as `set-<role>`. |
| `njobs` | `3` | Number of parallel workers used while reading headers and shapes. |
| `verbose` | `True` | If `True`, prints validation progress and detected shapes. |

Parameters of `prep.input_validation_gii`:

| Parameter | Default | Meaning |
|---|---|---|
| `file_list` | Required | List of hemisphere dictionaries. A two-hemisphere example is `{"l": left_file, "r": right_file}` for each subject/run. |
| `ses` | `"data"` | Session label written into output filenames as `ses-<label>`. |
| `bids_parse` | `True` | If `True`, parses BIDS-like filename entities from GIFTI filenames. |
| `subject_key` | `"sub"` | Prefix used to locate subject IDs when `bids_parse=False`. |
| `check_dim` | `True` | If `True`, checks dimensions within each hemisphere and group. |
| `group_by` | `"run"` | Filename entity used to group files before dimension checking. Use `None` to check all files together. |
| `role` | `"train"` | Dataset role written into output filenames as `set-<role>`. |
| `njobs` | `3` | Number of parallel workers used while reading headers and shapes. |
| `verbose` | `True` | If `True`, prints validation progress and detected shapes. |

Parameters of `prep.write_json_input_validate`:

| Parameter | Default | Meaning |
|---|---|---|
| `step_info` | Required | Validation dictionary returned by `prep.input_validation_nii` or `prep.input_validation_gii`. |
| `json_path` | Required | Path to the JSON file used to store validation records, usually `work_dir / "logs" / "prep_log.json"`. |
| `overwrite` | `False` | If `True`, starts a fresh validation log. If `False`, appends the validation record. |

The following cells define example CIFTI paths and subject IDs used by the validation examples. Replace all paths with paths on your machine.

In [ ]:
cifti_dir = Path("/path/to/raw_data/cifti")
work_dir = Path("/path/to/workdir_cifti")
json_path = work_dir / "logs" / "prep_log.json"
sub_list = ["sub-rid000005", "sub-rid000014", "sub-rid000029"]

For BIDS-like filenames, set `bids_parse=True`. The subject ID is read from the `sub` entity, and additional entities such as `run` are stored for later file filtering.

In [ ]:
data_list_bids = [
    cifti_dir / f"{sub}_ses-raiders_run-01_TS.dtseries.nii"
    for sub in sub_list
]

data_info_bids = prep.input_validation_nii(
    file_list=data_list_bids,
    ses="raiders",
    bids_parse=True,
    subject_key=None,
    check_dim=True,
    group_by="run",
    role="demo",
    njobs=2,
    verbose=True,
)

prep.write_json_input_validate(data_info_bids, json_path, overwrite=True)

For custom filenames, set `bids_parse=False` and provide the prefix that marks the subject identifier through `subject_key`.

In [ ]:
data_list = [
    cifti_dir / f"xSfrNGRd_{sub.split('-')[1]}_raiders_run-02_TS.dtseries.nii"
    for sub in sub_list
]

data_info = prep.input_validation_nii(
    file_list=data_list,
    ses="raiders",
    bids_parse=False,
    subject_key="xSfrNGRd",
    check_dim=True,
    group_by="run",
    role="demo",
    njobs=2,
    verbose=True,
)

prep.write_json_input_validate(data_info, json_path, overwrite=False)

### (2) Data Preprocessing

After validation, the preprocessing functions write prepared arrays under `work_dir / sub-<subject> / resample / response / <structure>`. Surface outputs use `hemi-L` and `hemi-R`; CIFTI-derived subcortical outputs use folders such as `volume-subcortical-L`; NIFTI ROI outputs use `volume-<roi_name>`.

`normalize` accepts `"zscore"`, `"demean"`, or `None`. When `normalize=None`, the data values are saved in raw form and the filename uses `desc-bold-raw`.

**CIFTI Cortical Data**

CIFTI cortical data are extracted from `.dtseries.nii` or `.dscalar.nii` files and saved as separate left and right cortical matrices. `prep.prep_surf` loads the CIFTI cortical data, patches the medial wall, applies surface mapping when requested, applies the target cortical or ROI mask, normalizes the matrix, and saves `.npy` files.

Parameters of `prep.prep_surf`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root output directory. Prepared files, logs, and masks are written inside this directory. |
| `input_info` | Required | Validation dictionary returned by `prep.input_validation_nii` or `prep.input_validation_gii`. |
| `do_resample` | `True` | If `True`, applies a surface mapper before masking. Use `False` when source and target spaces already match. |
| `res_param` | `None` | Dictionary describing the surface mapping and output resolution label. Required when `do_resample=True`. |
| `do_medial_rm` | `True` | If `True`, applies the mask specified by `medial_param`. This can remove the medial wall or extract a surface ROI. |
| `medial_param` | `None` | Dictionary describing the target-surface mask. Required when `do_medial_rm=True`. |
| `normalize` | `"zscore"` | Normalization method applied along the time axis for each vertex. Use `"zscore"`, `"demean"`, or `None`. |
| `dtype` | `np.float32` | Numeric dtype used while preparing and saving arrays. |
| `njobs` | `3` | Number of parallel workers used for file-wise preparation. |
| `verbose` | `5` | Joblib verbosity level. |
| `json_path` | `"prep_log.json"` | JSON path used to record preprocessing settings. |
| `overwrite` | `False` | If `True`, replaces the existing JSON record list for this write. If `False`, appends the record. |
| `log_num` | `"00001"` | Identifier appended to the progress log filename. |

Fields commonly used in `res_param` for surface data:

| Field | Default | Meaning |
|---|---|---|
| `source` | Required when resampling | Source surface space of the input data, such as `fslr-ico57`. |
| `source_mask` | `None` | Source-space mask used by the mapper. Provide a mask when source surface data have already had vertices removed. |
| `target` | Required when resampling | Target analysis space, such as `onavg-ico32`. |
| `target_mask` | `None` | Target-space mask used by the mapper. |
| `data` | `None` | Mapper object. With `None`, the function loads the configured built-in mapper. |
| `desc` | Required when logging | Human-readable resampling description written to logs. |
| `res` | Required for output names | Short resolution label written into output filenames as `res-<label>`. |

Fields commonly used in `medial_param` for surface data:

| Field | Default | Meaning |
|---|---|---|
| `space` | Required when loading a mask | Surface space used when loading a cortical mask through `neuroboros`. |
| `data` | `None` | Mask data. A dictionary such as `{"l": bool_array, "r": bool_array}` keeps vertices marked `True`. |
| `roi_name` | Required for output names | ROI label written into output filenames as `roi-<label>`. |
| `desc` | Required when logging | Human-readable mask description written to logs. |

The example below uses the configured mapper and cortical mask for `fslr-ico57` to `onavg-ico32` preparation.

In [ ]:
res_param = {
    "source": "fslr-ico57",
    "source_mask": None,
    "target": "onavg-ico32",
    "target_mask": None,
    "data": None,
    "desc": "fslr-ico57 whole to onavg-ico32 whole",
    "res": "onavg32",
}

medial_param = {
    "space": "onavg-ico32",
    "data": None,
    "roi_name": "cortical",
    "desc": "onavg-ico32 cortical",
}

prep.prep_surf(
        work_dir,
        input_info=data_info,
        do_resample=True,
        res_param=res_param,
        do_medial_rm=True,
        medial_param=medial_param,
        normalize="zscore",
        dtype=np.float32,
        njobs=3,
        verbose=5,
        json_path=json_path,
        overwrite=False,
        log_num='cifti_demo',
)


You can also provide your own mapper and target mask through `res_param["data"]` and `medial_param["data"]`.

In [ ]:
mapper_data = joblib.load("/path/to/mapper.pkl")
cortical_mask_data = joblib.load("/path/to/cortical_mask.pkl")

res_param = {
    "source": "fslr-ico57",
    "source_mask": None,
    "target": "onavg-ico32",
    "target_mask": None,
    "data": mapper_data,
    "desc": "fslr-ico57 whole to onavg-ico32 whole",
    "res": "onavg32",
}

medial_param = {
    "space": "onavg-ico32",
    "data": cortical_mask_data,
    "roi_name": "cortical",
    "desc": "onavg-ico32 cortical",
}

prep.prep_surf(
        work_dir,
        input_info=data_info,
        do_resample=True,
        res_param=res_param,
        do_medial_rm=True,
        medial_param=medial_param,
        normalize="zscore",
        dtype=np.float32,
        njobs=3,
        verbose=5,
        json_path=json_path,
        overwrite=False,
        log_num='cifti_demo',
)

**CIFTI Subcortical Data**

CIFTI subcortical data are extracted from the CIFTI BrainModels axis and saved as volume-derived `time x voxel` matrices. The standard GUI-compatible outputs are `volume-subcortical-L`, `volume-subcortical-R`, and `volume-subcortical-BRAIN_STEM`.

Parameters of `prep.prep_volume`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root output directory. |
| `input_info` | Required | Validation dictionary returned by `prep.input_validation_nii`. |
| `cifti_region` | `"whole"` | CIFTI subcortical selector. Common values are `"L"`, `"R"`, `"BRAIN_STEM"`, `"whole"`, a full CIFTI structure name, or a list of structures. |
| `roi_name` | `"cortical_cift"` | ROI label used in output folder names and filenames. |
| `do_resample` | `True` | If `True`, resamples the volume before masking. |
| `res_param` | `None` | Volume resampling dictionary. Required when `do_resample=True`. |
| `mask` | `None` | External NIFTI mask used for NIFTI input. CIFTI subcortical preparation uses the header-derived mask when this is `None`. |
| `save_resample_mask` | `True` | If `True`, saves a representative 3D mask in `work_dir / "masks"`. |
| `normalize` | `"zscore"` | Normalization method applied along the time axis for each voxel. Use `"zscore"`, `"demean"`, or `None`. |
| `dtype` | `np.float32` | Numeric dtype used while preparing and saving arrays. |
| `njobs` | `3` | Number of parallel workers used for file-wise preparation. |
| `verbose` | `5` | Joblib verbosity level. |
| `json_path` | `"prep_log.json"` | JSON path used to record preprocessing settings. |
| `overwrite` | `False` | If `True`, replaces the existing JSON record list for this write. If `False`, appends the record. |
| `log_num` | `"00001"` | Identifier appended to the progress log filename. |

Fields commonly used in `res_param` for volume data:

| Field | Default | Meaning |
|---|---|---|
| `ref` | Required for `mode="ref"` | Reference NIFTI image whose grid should be matched. |
| `interpolation` | `"nearest"` | Interpolation method passed to the volume resampling helper. Common values are `"nearest"`, `"linear"`, `"continuous"`, `"spline"`, and `"multilinear"`. |
| `vol_size` | `None` | Target voxel size used by voxel-size resampling and passed through reference-based workflows. |
| `mode` | `"ref"` | `"ref"` matches the grid of `ref`; `"vox"` creates a grid from `vol_size`. |
| `res` | `"nm"` when omitted | Short resolution label written into output filenames as `res-<label>`. |

For CIFTI subcortical preparation in the native CIFTI-derived grid, use `do_resample=False`, `res_param=None`, and `mask=None`.

In [ ]:
for structure_name in ["L", "R", "BRAIN_STEM"]:
    prep.prep_volume(
        work_dir,
        input_info=data_info,
        cifti_region=structure_name,
        roi_name=f"subcortical-{structure_name}",
        do_resample=False,
        res_param=None,
        mask=None,
        save_resample_mask=True,
        normalize="zscore",
        dtype=np.float32,
        njobs=2,
        verbose=5,
        json_path=json_path,
        overwrite=False,
        log_num="cifti_subcortical",
    )

**GIFTI Cortical Data**

GIFTI preparation is surface-only. Each entry in `file_list` is a dictionary whose keys identify hemispheres. Use `{"l": left_gifti, "r": right_gifti}` when both hemispheres are present. After `prep.input_validation_gii`, use `prep.prep_surf` with the same surface parameter structure shown above.

Prepared GIFTI outputs are saved in `hemi-L` and `hemi-R` folders, matching the structure names used by downstream hyperalignment functions.

In [ ]:
gifti_dir = Path("/path/to/raw_data/gifti")
work_dir = Path("/path/to/workdir_gifti")
json_path = work_dir / "logs" / "prep_log.json"

sub_list = [
    "sub-rid000005",
    "sub-rid000014",
    "sub-rid000029",
]

train_list = []
for sub in sub_list:
    train_list.append(
        {
            "l": gifti_dir / f"{sub}_task-raiders_acq-8ch336vol_run-01_hemi-L_space-fsaverage_bold.func.gii",
            "r": gifti_dir / f"{sub}_task-raiders_acq-8ch336vol_run-01_hemi-R_space-fsaverage_bold.func.gii",
        }
    )

gifti_info = prep.input_validation_gii(
    file_list=train_list,
    ses="raiders",
    bids_parse=True,
    subject_key=None,
    check_dim=True,
    group_by="run",
    role="train",
    njobs=2,
    verbose=True,
)

prep.write_json_input_validate(gifti_info, json_path, overwrite=False)

res_param = {
    "source": "fslr-ico57",
    "source_mask": None,
    "target": "onavg-ico32",
    "target_mask": None,
    "data": None,
    "desc": "fslr-ico57 whole to onavg-ico32 whole",
    "res": "onavg32",
}

medial_param = {
    "space": "onavg-ico32",
    "data": None,
    "roi_name": "cortical",
    "desc": "onavg-ico32 cortical",
}

prep.prep_surf(
    work_dir,
    input_info=gifti_info,
    do_resample=True,
    res_param=res_param,
    do_medial_rm=True,
    medial_param=medial_param,
    normalize="zscore",
    dtype=np.float32,
    njobs=3,
    verbose=5,
    json_path=json_path,
    overwrite=False,
    log_num="gifti_demo",
)

**NIFTI Volume Data**

NIFTI volume preparation uses `prep.input_validation_nii` followed by `prep.prep_volume`. Features are selected with a 3D NIFTI mask when `mask` is provided. The `roi_name` value becomes the output structure folder, for example `volume-cortical` or `volume-mPFC`.

Set `cifti_region=None` for NIFTI input. Use `res_param` to match a reference image or to resample by voxel size before masking.

In [ ]:
nifti_dir = Path("/path/to/raw_data/nifti")
work_dir = Path("/path/to/workdir_nifti")
json_path = work_dir / "logs" / "prep_log.json"

sub_list = [
    "sub-rid000005",
    "sub-rid000014",
    "sub-rid000029",
]

nifti_list = [
    nifti_dir / f"{sub}_task-raiders_acq-8ch336vol_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
    for sub in sub_list
]

nifti_info = prep.input_validation_nii(
    file_list=nifti_list,
    ses="raiders",
    bids_parse=True,
    subject_key=None,
    check_dim=True,
    group_by="run",
    role="train",
    njobs=2,
    verbose=True,
)

prep.write_json_input_validate(nifti_info, json_path, overwrite=False)

res_param = {
    "ref": "/path/to/HCPex_2mm_cortical_mask.nii",
    "interpolation": "nearest",
    "vol_size": 3,
    "mode": "ref",
    "res": "3mm",
}

mask = "/path/to/HCPex_2mm_cortical_mask.nii"

prep.prep_volume(
    work_dir,
    input_info=nifti_info,
    cifti_region=None,
    roi_name="cortical",
    do_resample=True,
    res_param=res_param,
    mask=mask,
    save_resample_mask=True,
    normalize="zscore",
    dtype=np.float32,
    njobs=2,
    verbose=5,
    json_path=json_path,
    overwrite=False,
    log_num="nifti_demo",
)

**Concatenating Prepared Files**

After preparing individual runs or modules, use `prep.combine_file` to concatenate matching `.npy` files along the time dimension. Concatenation is performed separately for each subject and requested structure.

Parameters of `prep.combine_file`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root directory containing subject folders. |
| `sub_list` | Required | Subject folder names to process, such as `"sub-rid000005"`. |
| `step` | Required | Processing step folder to search, usually `"resample"`. |
| `modality` | Required | Modality folder under `step`, commonly `"response"` or `"connectivity"`. |
| `structures` | Required | Structure folder or list of folders to concatenate, such as `"hemi-L"` or `"volume-subcortical-L"`. |
| `ses_name` | Required | Session label assigned to the concatenated output filename. |
| `njobs` | Required | Number of parallel workers across subject-structure jobs. |
| `method` | `"zscore"` | Optional normalization applied after concatenation. Use `None` to save concatenated values directly. |
| `json_path` | `"prep_log.json"` | JSON path used to record the concatenation step. |
| `dtype` | `"np.float32"` | Output numeric dtype used when post-concatenation normalization is applied. |
| `overwrite` | `False` | If `True`, replaces the existing JSON record list for this write. If `False`, appends the record. |
| `log_num` | `"00001"` | Identifier appended to the progress log filename. |
| `**file_filter` | Empty | BIDS-like filename filters used to select files, such as `run=["01", "02"]`, `set="train"`, or `roi="cortical"`. |

`file_filter` is matched against filename entities. Add filters such as `space`, `roi`, `res`, `desc`, or `set` when a folder contains multiple candidate files.

In [ ]:
work_dir = Path("/path/to/workdir_cifti")
json_path = work_dir / "logs" / "prep_log.json"

sub_list = [
    "sub-rid000005",
    "sub-rid000014",
    "sub-rid000029",
]

file_filter = {
    "run": ["01", "02"],
}

prep.combine_file(
    work_dir,
    sub_list,
    step="resample",
    modality="response",
    structures=[
        "hemi-L",
        "hemi-R",
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    ses_name="comb12",
    njobs=2,
    method=None,
    json_path=json_path,
    dtype=np.float32,
    overwrite=False,
    log_num="combine_runs",
    **file_filter,
)

## GUI Pipeline

Launch the graphical interface with `gui()`. After the main window opens, complete data preparation by filling the GUI fields, saving the configuration when needed, and clicking `RUN!`.

In [ ]:
from fmriha.gui import gui
gui()

```{image} pic/data_prep/gui.png
:width: 800px
```

At the very top of the main page is the output directory, **Work Dir**. You can either click the **···** button to open the file manager and select the corresponding directory, or directly enter your output path in the input box.

```{image} pic/data_prep/gui_workdir.png
:width: 800px
```

If you want to process Surface data, please click the **Space and Searchlight Configuration** button and select **Space** in the **Surface** module. We recommend choosing **onavg-ico32** here.

```{image} pic/data_prep/gui_space.png
:width: 800px
```

Next, click the **Data Preparation** button on the main page. On this page, you can provide different datasets for different modules. You can decide which data to use for the Template, calculating the Transformation Matrix, and applying the Transformation Matrix (Alignment).

If you want to provide data for template computation, you need to check the box in front of Template. The same applies to Transformation Data and Aligned Data.

```{image} pic/data_prep/gui_datachoose.png
:width: 800px
```

The following operations will use Template data as an example.

We support three types of data: **CIFTI**, **GIFTI**, and **NIFTI**, which can be selected under **Data Type**.

If you choose **CIFTI**, the available options in **Structure** include **hemi-L**, **hemi-R**, **subcortical-L**, **subcortical-R**, and **subcortical-brain_stem**. Among them, **hemi-L** and **hemi-R** correspond to cortical surface data, while **subcortical-L**, **subcortical-R**, and **subcortical-brain_stem** correspond to subcortical volumetric data.

Next, you can either click the **Add Files** button to open the file manager and select the desired files, or directly enter the corresponding file paths into the table.

If your **Structure** is set to the surface types **hemi-L** or **hemi-R**, you also need to specify the original space of the imported data in **Surface** (for example, **fslr-ico57** in the example below). In addition, if the surface portion of your CIFTI file has already had the medial wall removed, you need to provide the corresponding medial wall file in **Source Medial Wall Removed**.

Further down, you can choose the normalization method for the data (**raw** means no normalization will be applied). If your filenames follow the BIDS convention, simply check **BIDS File**. Otherwise, you need to fill in the **Subject Key** to specify the subject identifier (for example, **xSfrNGRd** in the example below).

```{image} pic/data_prep/gui_cifti.png
:width: 800px
```

If you select **GIFTI** as the **Data Type**, the workflow is very similar to that of **CIFTI**. The only difference is that you need to use **Add Files (hemi-L)** and **Add Files (hemi-R)** to load the left- and right-hemisphere files separately.

If you select **NIFTI** as the **Data Type**, you need to choose **volume-ROI** in **Structure** and fill in the corresponding information in the **Volume (NIFTI)** module. Among them, **Mask** refers to the corresponding `.nii` file.

```{image} pic/data_prep/gui_nifti.png
:width: 800px
```

If you want to use files from multiple runs, you can click the "+" button at the lower right corner of each module, and then check the **Combine Files** option at the bottom.

```{image} pic/data_prep/gui_combine.png
:width: 800px
```

For Transformation Data and Aligned Data, if you want to directly reuse the files from the previous modules, you can check the corresponding **Same as** options. Here, **Same as Tmpl. Data** means using the same data as the Template module, while **Same as T Data** means using the same data as the Transformation Matrix module. After clicking a **Same as** option, you will no longer be able to modify the contents within the current module. Any changes must be made in the source module that the settings were copied from.

In addition, the **Reset All Settings** button on the **Data Preparation** page only clears the settings within the module where the button is located; it will **not** affect the settings of other modules.